# Biaya masuk sekolah: seberat apa Juli?

**Abstrak ringkas.** Menjelang tahun ajaran baru, perbincangan tentang "biaya masuk sekolah" terpusat pada uang pangkal dan pendaftaran. Edisi ini menguji framing tersebut dengan data Susenas MSBP tahun ajaran 2023/2024: enam belas komponen biaya per murid per tahun untuk empat jenjang pendidikan dikelompokkan ke dalam lima golongan dan diterjemahkan ke rupiah. Tiga temuan utama: (i) rerata biaya setahun naik tidak linier lintas jenjang, dari Rp4,56 juta (SD) menjadi Rp19,01 juta (pendidikan tinggi); (ii) komponen awal tahun (pendaftaran, seragam, buku, LKS, alat tulis) hanya menjangkau sekitar 12-16 persen beban di SD-SMA, sedangkan uang saku dan transport menelan sekitar tiga per empat; (iii) biaya di keluarga 20% berpengeluaran teratas adalah 2,71 kali keluarga 40% terbawah pada jenjang SD, dengan kesenjangan yang bergerak menurun hingga jenjang menengah. Karena data ini mengabadikan keadaan sebelum Putusan MK No. 3/PUU-XXIII/2025 yang mewajibkan jenjang SD-SMP tanpa pungutan, edisi ini terbaca sekaligus sebagai garis pangkal (baseline) untuk menilai pelaksanaannya kelak.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from tools import gaya

gaya.terapkan()

JENJANG = ["sd", "smp", "sma_smk", "pt"]
LABEL = {"sd": "SD", "smp": "SMP", "sma_smk": "SMA/SMK", "pt": "PT"}

total = pd.read_csv("data/spp-total-karakteristik.csv").set_index("kelompok")
kom = pd.read_csv("data/spp-proporsi-komponen.csv").set_index("komponen")
T = total.loc["total"]  # juta rupiah per murid per tahun
print("kelompok karakteristik:", list(total.index))
print("jumlah komponen:", len(kom))

kelompok karakteristik: ['total', 'kelompok_20_atas', 'kelompok_40_tengah', 'kelompok_40_bawah', 'perkotaan', 'perdesaan']
jumlah komponen: 16


## Latar Belakang

Setiap bulan Juli, keluarga Indonesia memasuki tahun ajaran baru dan diskursus publik kembali dipenuhi satu frasa: *biaya masuk sekolah*. Frasa itu biasanya dimaksudkan sebagai uang pangkal atau uang pendaftaran, dan ia membentuk anggapan pinjaman bahwa beban pendidikan adalah beban musiman yang terpusat di awal tahun. Per 27 Mei 2025 percakapan itu berubah status: Putusan MK No. 3/PUU-XXIII/2025 menyatakan Pasal 34 ayat (2) UU Sisdiknas hanya konstitusional bila dimaknai menjamin pendidikan dasar tanpa memungut biaya, baik pada satuan yang diselenggarakan pemerintah maupun masyarakat [2]. Putusan tersebut memerlukan potret "sebelum" yang jujur: seberapa besar sebenarnya beban biaya pendidikan keluarga per tahun, dan komponen mana saja yang menyusunnya.

Edisi ini menguji tiga pertanyaan: (1) berapa rerata biaya pendidikan satu tahun ajaran penuh per murid pada tiap jenjang; (2) berapa porsi komponen yang layak disebut "biaya awal tahun" dibandingkan beban yang berjalan tiap hari; dan (3) seberapa timpang bebannya antar kelompok pengeluaran rumah tangga serta antara wilayah perkotaan dan perdesaan.

## Data & Metode

Sumber tunggal ialah publikasi BPS "Statistik Penunjang Pendidikan 2024" (Katalog 4301007) [1], hasil pengolahan Susenas Modul Sosial Budaya dan Pendidikan (MSBP) untuk tahun ajaran 2023/2024. Dua tabel dipakai dan diekstrak terprogram dari PDF publikasi (`tools/data_scraping.py`, dengan sahakan per sel dan uji jumlah kolom 100 +/- 0,2 poin persen):

- **Tabel 2.1**: rata-rata total biaya pendidikan per murid per tahun menurut karakteristik (kelompok pengeluaran rumah tangga 20 persen teratas, 40 persen menengah, 40 persen terbawah; perkotaan dan perdesaan) kali empat jenjang: SD, SMP, SMA/SMK, pendidikan tinggi (PT). Satuan juta rupiah.
- **Tabel 2.2**: proporsi enam belas komponen biaya menurut jenjang, dari uang pendaftaran sampai uang transpor, dinyatakan dalam persen.

Enam belas komponen dikelompokkan ke dalam lima golongan yang didefinisikan di muka: **(a) awal tahun ajaran** (pendaftaran, seragam dan perlengkapannya, buku, LKS, alat tulis); **(b) iuran dan kegiatan sekolah** (SPP/UKT, komite, ekstrakurikuler, praktikum, kursus sekolah, evaluasi, study tour, pendukung pembelajaran); **(c) uang saku**; **(d) uang transpor**; dan **(e) lainnya**. Terjemahan ke rupiah dihitung sebagai porsi komponen dikali total rerata Tabel 2.1 per jenjang: taksiran komposit atas rerata nasional, bukan tagihan satu rumah tangga tertentu.

Ketidakpastian pengukuran diperlakukan dua langkah. Pertama, galat contoh untuk rancangan sampel telah diterbitkan BPS pada Tabel 2.13 s.d. 2.16 publikasi yang sama [1]; keberadaannya dikutip alih-alih dihitung ulang, karena data mikro tidak diakses. Kedua, klaim antar kelompok hanya dibaca bila besarnya orde faktor (selisih dua kali lipat atau lebih), jauh di atas orde galat contoh tipikal pada tabel-tabel demikian.

In [2]:
# verifikasi kontrak data + hitung semua ukuran turunan
assert len(total) == 6 and len(kom) == 16
for j in JENJANG:
    assert abs(kom[j].sum() - 100) <= 0.2, j
# kunci angka publikasi (Tabel 2.1)
assert list(T.round(2)) == [4.56, 7.34, 10.19, 19.01]

AWAL = ["Uang Pendaftaran", "Baju Sekolah dan Perlengkapannya",
        "Buku Pelajaran/Panduan/Diktat", "Lembar Kerja Siswa (LKS)",
        "Alat Tulis dan Perlengkapan Lainnya"]
IURAN = ["SPP/Uang Kuliah Tunggal (UKT)", "Komite Sekolah", "Ekstrakurikuler",
         "Praktikum/Keterampilan dan Bahan Penunjangnya",
         "Kursus yang Diselenggarakan Sekolah", "Evaluasi/Ujian",
         "Kunjungan Edukatif (Study Tour)", "Pendukung Pembelajaran"]
gol = pd.DataFrame({
    "awal_tahun": kom.loc[AWAL].sum(),
    "iuran_sekolah": kom.loc[IURAN].sum(),
    "uang_saku": kom.loc["Uang Saku"],
    "uang_transpor": kom.loc["Uang Transpor"],
    "lainnya": kom.loc["Lainnya"],
})
assert (gol.sum(axis=1) - 100).abs().max() <= 0.2  # per jenjang

rp = gol.mul(T, axis=0) / 100  # juta rupiah per tahun
perbulan = rp / 12 * 1000      # ribu rupiah per bulan
assert rp.notna().all().all()
rasio_ab = total.loc["kelompok_20_atas"] / total.loc["kelompok_40_bawah"]
rasio_kd = total.loc["perkotaan"] / total.loc["perdesaan"]

print("porsi golongan (%):")
print(gol.round(1))
print("taksiran rupiah per tahun (juta):")
print(rp.round(2))
print("uang saku+transpor per bulan (ribu):")
print((perbulan.uang_saku + perbulan.uang_transpor).round(0))
print("rasio 20% atas : 40% bawah =", rasio_ab.round(2).to_dict())
print("rasio kota : desa          =", rasio_kd.round(2).to_dict())
pend = kom.loc["Uang Pendaftaran"].mul(T, axis=0) / 100
print("uang pendaftaran (juta):", pend.round(3).to_dict())

porsi golongan (%):
         awal_tahun  iuran_sekolah  uang_saku  uang_transpor  lainnya
sd             16.3            7.7       61.4           13.6      0.9
smp            14.4            9.4       55.9           19.1      1.3
sma_smk        12.6           13.6       51.0           22.1      0.8
pt              7.9           38.0       35.1           18.2      0.7
taksiran rupiah per tahun (juta):
         awal_tahun  iuran_sekolah  uang_saku  uang_transpor  lainnya
sd             0.75           0.35       2.80           0.62     0.04
smp            1.06           0.69       4.10           1.40     0.10
sma_smk        1.28           1.38       5.19           2.25     0.08
pt             1.50           7.23       6.68           3.47     0.14
uang saku+transpor per bulan (ribu):
sd         285.0
smp        459.0
sma_smk    621.0
pt         846.0
dtype: float64
rasio 20% atas : 40% bawah = {'sd': 2.71, 'smp': 2.05, 'sma_smk': 1.9, 'pt': 1.96}
rasio kota : desa          = {'sd': 1.59, '

## Hasil

Tiga fakta utama. Pertama, tingkat biaya: rerata nasional per murid per tahun adalah Rp4,56 juta (SD), Rp7,34 juta (SMP), Rp10,19 juta (SMA/SMK), dan Rp19,01 juta untuk pendidikan tinggi, yakni kira-kira 1,6; 1,4; dan 1,9 kali lipat tiap lompatan jenjang. Kedua, struktur: komponen yang diandaikan sebagai "biaya masuk" oleh percakapan publik hanya sebagian kecil dari beban sesungguhnya. Uang pendaftaran saja 2,13 persen (SD) sampai 5,05 persen (PT) dari total; seluruh paket awal tahun ajaran menjangkau 12-16 persen di SD-SMA dan hanya 7,88 persen di PT. Yang dominan justru konsumsi harian sekolah: uang saku 50,98-61,44 persen di SD-SMA dan transport 13,60-22,11 persen, gabungan keduanya 73-75 persen di enam belas komponen sekolah dasar dan menengah. Ketiga, kesenjangan: biaya di keluarga 20 persen berpengeluaran teratas adalah 2,71 kali lipat biaya di keluarga 40 persen terbawah pada jenjang SD; rasio ini turun ke 2,05 (SMP) dan 1,90 (SMA/SMK), lalu 1,96 di PT. Kesenjangan kota-desa mengecil dari 1,59 (SD) ke 1,18 (SMA/SMK) sebelum kembali 1,33 di PT.

In [3]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "'Biaya masuk sekolah': berapa porsinya dari biaya setahun?",
        "Paket awal tahun hanya 12-16 persen di SD-SMA; lainnya dominan biaya harian: saku & transpor.",
        "BPS, Statistik Penunjang Pendidikan 2024, Tabel 2.1 & 2.2 (Susenas MSBP, TA 2023/2024; ujung batang: total per tahun)",
        16 * 4,
    )
    warna = {"awal_tahun": gaya.AKSEN, "iuran_sekolah": "#5A6B8C",
             "uang_saku": gaya.INK, "uang_transpor": "#8E99B3", "lainnya": "#C9C4B8"}
    labelgor = {"awal_tahun": "awal tahun ajaran", "iuran_sekolah": "iuran & kegiatan sekolah",
                "uang_saku": "uang saku", "uang_transpor": "uang transpor", "lainnya": "lainnya"}
    nampil = {"awal_tahun": "white", "iuran_sekolah": "white", "uang_saku": "white",
              "uang_transpor": "white", "lainnya": gaya.INK}
    y = np.arange(len(JENJANG))
    kiri = np.zeros(len(JENJANG))
    for g in ["awal_tahun", "iuran_sekolah", "uang_saku", "uang_transpor", "lainnya"]:
        v = gol.loc[JENJANG, g].to_numpy(dtype=float)
        ax.barh(y, v, left=kiri, color=warna[g], height=0.66, label=labelgor[g], zorder=3)
        for i, (vv, ll) in enumerate(zip(v, kiri)):
            if vv >= 6:
                ax.text(ll + vv / 2, i, gaya.idn(vv, 1), ha="center", va="center",
                        fontsize=8.6 * fs, color=nampil[g], zorder=4,
                        fontweight="bold" if g == "awal_tahun" else "normal")
        kiri += v
    for i, j in enumerate(JENJANG):
        ax.text(101.2, i, f"Rp{gaya.idn(T[j], 2)} jt", va="center", fontsize=8.6 * fs,
                color=gaya.INK2)
    ax.set_yticks(y)
    ax.set_yticklabels([LABEL[j] for j in JENJANG], fontsize=9.5 * fs)
    ax.invert_yaxis()
    ax.set_xlim(0, 114)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("porsi biaya per murid per tahun (%)", fontsize=9.5 * fs)
    ax.legend(fontsize=8.4 * fs, loc="lower center", bbox_to_anchor=(0.5, -0.34),
              ncol=3, frameon=False, columnspacing=1.1, handlelength=1.2)
    return fig


for mode in ("readme", "linkedin"):
    gaya.simpan(grafik_1(mode), "01-komposisi-biaya", mode)
print("grafik 1 tersimpan (2 mode)")

grafik 1 tersimpan (2 mode)


Gambar 1 menyusun ulang narasi "Juli yang mahal": oranye (awal tahun ajaran) memang nyata tetapi sempit, sedangkan warna gelap (uang saku) membentuk mayoritas setiap batang sekolah dasar dan menengah; perguruan tinggi memperlihatkan struktur ketiga di mana iuran (UKT) menjadi blok terbesar kedua setelah uang saku. Pengamatan terpisah: total nominal bertambah lintas jenjang, tetapi porsi harian berkurang, sehingga pertanyaan kebijakan dan pertanyaan anggaran keluarga bergeser dari "berani membayar berapa di depan" menjadi "sanggup menahan berapa per bulan".

In [4]:
def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Apakah biaya sekolah setahun sama beratnya bagi semua keluarga?",
        "Di SD, keluarga 20% teratas membelanjakan 2,71 kali biaya keluarga 40% terbawah; PT 1,96 kali.",
        "BPS, Statistik Penunjang Pendidikan 2024, Tabel 2.1 (Susenas MSBP, TA 2023/2024)",
        3 * 4,
    )
    kel = ["kelompok_40_bawah", "kelompok_40_tengah", "kelompok_20_atas"]
    warna3 = ["#9AA3B6", "#5A6B8C", gaya.INK]
    leg3 = ["40% pengeluaran terbawah", "40% pengeluaran menengah", "20% pengeluaran teratas"]
    x = np.arange(len(JENJANG))
    lebar = 0.26
    for k, (slug, w3, lg) in enumerate(zip(kel, warna3, leg3)):
        v = total.loc[slug, JENJANG].to_numpy(dtype=float)
        pos = x + (k - 1) * lebar
        ax.bar(pos, v, width=lebar, color=w3, label=lg, zorder=3)
        for px, vv in zip(pos, v):
            ax.text(px, vv + 0.45, gaya.idn(vv, 2), ha="center", fontsize=8.0 * fs,
                    color=gaya.INK2)
    for i, j in enumerate(JENJANG):
        atas = total.loc["kelompok_20_atas", j]
        ax.annotate(f"{gaya.idn(rasio_ab[j], 2)}x", (x[i], atas + 2.1), ha="center",
                    fontsize=9.2 * fs, color=gaya.AKSEN, fontweight="bold")
    ax.annotate(chr(10).join(["rasio 20% teratas", "dibanding 40% terbawah"]),
                (0.05, 0.60), ha="left", xycoords="axes fraction",
                fontsize=7.8 * fs, color=gaya.AKSEN, linespacing=1.4)
    ax.set_xticks(x)
    ax.set_xticklabels([LABEL[j] for j in JENJANG], fontsize=9.5 * fs)
    ax.set_ylim(0, 30)
    ax.set_ylabel("biaya per murid per tahun (juta rupiah)", fontsize=9.5 * fs)
    ax.legend(fontsize=8.4 * fs, loc="upper left", frameon=False)
    return fig


for mode in ("readme", "linkedin"):
    gaya.simpan(grafik_2(mode), "02-biaya-antar-keluarga", mode)
print("grafik 2 tersimpan (2 mode)")

grafik 2 tersimpan (2 mode)


In [5]:
# kunci angka untuk README & posting
metrik = {
    "edisi": 12,
    "sumber": "BPS, Statistik Penunjang Pendidikan 2024 (Katalog 4301007), Susenas MSBP pada tahun ajaran 2023/2024",
    "total_juta_per_tahun": {j: round(float(T[j]), 2) for j in JENJANG},
    "porsi_persen": {j: {g: round(float(gol.loc[j, g]), 2) for g in gol.columns} for j in JENJANG},
    "taksiran_rupiah_juta_per_tahun": {j: {g: round(float(rp.loc[j, g]), 3) for g in rp.columns} for j in JENJANG},
    "uang_pendaftaran_juta": {j: round(float(pend[j]), 3) for j in JENJANG},
    "saku_transpor_persen": {j: round(float(gol.loc[j, "uang_saku"] + gol.loc[j, "uang_transpor"]), 2) for j in JENJANG},
    "saku_ribu_per_bulan": {j: round(float(perbulan.loc[j, "uang_saku"])) for j in JENJANG},
    "rasio_atas_per_bawah": {j: round(float(rasio_ab[j]), 2) for j in JENJANG},
    "rasio_kota_per_desa": {j: round(float(rasio_kd[j]), 2) for j in JENJANG},
    "kelompok_pengeluaran": {k: {j: round(float(total.loc[k, j]), 2) for j in JENJANG}
                             for k in ["kelompok_20_atas", "kelompok_40_tengah", "kelompok_40_bawah",
                                       "perkotaan", "perdesaan"]},
    "catatan": [
        "Galat sampel tersedia pada Tabel 2.13 s.d. 2.16 publikasi; tidak dihitung ulang tanpa data mikro.",
        "Terjemahan rupiah adalah porsi komponen x total rerata per jenjang: taksiran komposit.",
        "Data mendahului Putusan MK No. 3/PUU-XXIII/2025 (27 Mei 2025); dipakai sebagai garis pangkal.",
    ],
}
Path("data/metrik-012.json").write_text(json.dumps(metrik, indent=1, ensure_ascii=False))
print("metrik-012.json tersimpan")

metrik-012.json tersimpan


## Pembahasan

Hasil-hasil di atas membaca ulang frasa "biaya masuk sekolah" dalam dua arah. Arah pertama: secara nominal, murid memang membayar lebih di depan; uang pendaftaran tumbuh dari taksiran Rp0,10 juta (SD) menjadi Rp0,25 juta (SMP), Rp0,40 juta (SMA/SMK), dan Rp0,96 juta (PT), sementara paket awal tahun seluruhnya mencapai kira-kira Rp0,75 juta (SD), Rp1,06 juta (SMP), dan Rp1,28 juta (SMA/SMK). Arah kedua sekaligus mengoreksi framing: paket itu hanyalah seperdelapan hingga seperenam beban setahun di sekolah dasar dan menengah, dan porsinya susut justru ketika jenjang naik. Beban terbesar keluarga bukanlah *masuk*, melainkan *menetap*: uang saku taksiran Rp233 ribu per bulan (SD) hingga Rp433 ribu per bulan (SMA/SMK), ditambah transport Rp52-188 ribu per bulan. Dengan kata lain, bantah "sekolah gratis" yang ujar-praktis dihadapi orang tua tidak berdiri pada papan iuran (7,70 persen di SD, di dalamnya SPP 4,88 persen dan komite 1,13 persen), melainkan pada struk jajan dan ongkos.

Dimensi kedua adalah keadilan antar keluarga. Kesenjangan biaya terbesar berada di jenjang yang paling seharusnya ringan: 2,71 kali lipat di SD. Bila beban dibaca terhadap kelompok pengeluaran, keluarga 40 persen terbawah tetap mengeluarkan Rp3,28 juta per anak per tahun untuk SD; pengalihan Rp0,27 juta senilai SPP dan komite SD tentu menolong, tetapi tidak menyentuh tiga perempat beban yang tinggal. Kesenjangan yang menurun lintas jenjang ada dua pembacaan yang tidak saling meniadakan: biaya jenjang lanjut naik untuk semua kelompok (mengecilkan rasio), dan seleksi masuk jenjang lanjut menyaring kelompok terbawah sebelum sempat membandingkan belanja (Tabel ini hanya merekam keluarga yang anaknya bersekolah).

Ketiga, edisi ini berfungsi sebagai garis pangkal kebijakan. Data Susenas MSBP berikutnya, sesuai siklusnya yang tiga tahunan, akan memungkinkan uji prediksi terukur atas Putusan MK No. 3/PUU-XXIII/2025 [2]: porsi uang pendaftaran, SPP, dan komite seharusnya mengecil menuju nol pada jenjang SD-SMP, sementara porsi uang saku dan transport, berada di luar jangkauan putusan, seharusnya tidak bergeser sistematis. Angka edisi ini adalah titik tolak perbandingan itu.

## Batasan

Pertama, modul MSBP bertanya biaya yang dikeluarkan dalam satu tahun ajaran menurut pelaporan responden; angkanya adalah rerata per murid dan dapat memuat bias ingatan serta samar batas apa yang dicatat keluarga sebagai biaya pendidikan. Kedua, rerata nasional tidak membedakan satuan negeri dan swasta maupun kabupaten/kota, sedangkan publikasi sumber menunjukkan beda tersebut substansial (tabel lain pada publikasi yang sama menyediakannya; di luar jangkauan edisi ini). Ketiga, definisi "awal tahun ajaran" di sini adalah operasionalisasi atas komponen BPS; komponen bernama lain dalam praktik sekolah, seperti sumbangan pengembangan pada sebagian satuan swasta, belum tentu terpetakan rapi ke dalamnya. Keempat, terjemahan porsi kali total adalah taksiran komposit yang menyamaratakan, bukan tagihan rumah tangga tertentu. Kelima, tahun ajaran 2023/2024 mendahului Putusan MK No. 3/PUU-XXIII/2025; simpulan apa pun tentang efek putusan adalah spekulasi di luar data ini. Keenam, perbandingan antar jenjang dan antar kelompok bersifat deskriptif, bukan uji statistik; galat sampel tersedia pada Tabel 2.13 s.d. 2.16 sumber [1].

## Kesimpulan

Terhadap pertanyaan "seberat apa biaya masuk sekolah", jawaban data bersyarat namun tegas. Beban awal tahun riil tetapi porsinya kecil: pendaftaran 2,13-5,05 persen dan paket awal tahun 12-16 persen dari beban setahun di SD-SMA. Beban yang justru menentukan adalah harian: uang saku dan transport memakan sekitar tiga per empat biaya pendidikan dasar dan menengah. Kesenjangan terlebar justru pada jenjang termuda (2,71 kali lipat di SD) dan tetap besar pada PT, di mana SPP/UKT menjadi blok terbesar (33,47 persen). Dengan Putusan MK No. 3/PUU-XXIII/2025 yang menjamin SD-SMP tanpa pungutan, angka-angka ini berdiri sebagai garis pangkal untuk pertanyaan berikut: apakah siklus data berikutnya menunjukkan porsi iuran menuju nol tanpa pergeseran liar ke komponen lain.

## Referensi

[1] Badan Pusat Statistik, *Statistik Penunjang Pendidikan 2024*, Katalog 4301007, Jakarta, 2025. Tersedia: https://www.bps.go.id/publication (Susenas MSBP, tahun ajaran 2023/2024; Tabel 2.1, 2.2, 2.13-2.16).
[2] Mahkamah Konstitusi RI, *Putusan Nomor 3/PUU-XXIII/2025*, pengujian Pasal 34 ayat (2) UU No. 20 Tahun 2003 tentang Sistem Pendidikan Nasional, dibacakan 27 Mei 2025.